<center>
    <p style="text-align:center">
        <a href="https://docs.futureagi.com/">Docs</a>
        |
        <a href="https://github.com/future-agi/traceAI">GitHub</a>
        |
        <a href="https://futureagi.com/">Website</a>
    </p>
</center>
<h1 align="center">Tracing the Anthropic SDK with Future AGI</h1>

This cookbook is a minimal quick-start showing how to wire [Future AGI](https://futureagi.com) tracing into the Anthropic Python SDK using the [`traceAI-anthropic`](https://pypi.org/project/traceAI-anthropic/) package. After two lines of setup, every subsequent call to `client.messages.create(...)` is captured as a span in your Future AGI project — including the prompt, the model parameters, the tool calls Claude returns, the token usage, and the latency.

## What is Future AGI?

[Future AGI](https://futureagi.com) is an observability and evaluation platform for AI agents and LLM applications. The [`traceAI`](https://github.com/future-agi/traceAI) open-source library provides auto-instrumentation packages for the Anthropic SDK, the Mistral SDK, LiteLLM, Google ADK, agno, DSPy, and other frameworks — all built on OpenTelemetry.

## Setup

### Install dependencies

> **Requires Python 3.10 or newer** — `traceAI-anthropic` (via `fi-instrumentation-otel`) uses PEP 604 union syntax (`X | Y`) which only parses on Python 3.10+. The default Colab runtime already meets this.

`traceai-anthropic` declares `anthropic>=0.34` as a runtime dependency, so the SDK installs transitively.

In [ ]:
%pip install -q traceai-anthropic

### Set environment variables

Replace the placeholders below with your real keys, then run the cell.

- **Future AGI** — sign up at [app.futureagi.com](https://app.futureagi.com) and copy `FI_API_KEY` and `FI_SECRET_KEY` from the dashboard.
- **Anthropic** — create a key at [console.anthropic.com/settings/keys](https://console.anthropic.com/settings/keys).

In [ ]:
import os

# Future AGI credentials — https://app.futureagi.com
os.environ["FI_API_KEY"] = os.getenv("FI_API_KEY", "fi-xxx")
os.environ["FI_SECRET_KEY"] = os.getenv("FI_SECRET_KEY", "fi-secret-xxx")

# Anthropic credentials — https://console.anthropic.com/settings/keys
os.environ["ANTHROPIC_API_KEY"] = os.getenv("ANTHROPIC_API_KEY", "sk-ant-xxx")

### Register the Future AGI tracer and instrument the Anthropic SDK

Call `register(...)` once to create a tracer provider that exports spans to Future AGI, then attach `AnthropicInstrumentor` so every subsequent call to the Anthropic SDK is captured automatically.

In [ ]:
from fi_instrumentation import register
from fi_instrumentation.fi_types import ProjectType
from traceai_anthropic import AnthropicInstrumentor

trace_provider = register(
    project_type=ProjectType.OBSERVE,
    project_name="anthropic_quickstart",
)
AnthropicInstrumentor().instrument(tracer_provider=trace_provider)

## Example: a single tool call

We send one user message to Claude and expose one `get_weather` tool. Claude picks the tool, fills in the arguments, and returns a `tool_use` block. We don't run the agent loop here — the point is to show that **one** call to `client.messages.create(...)` produces a fully-attributed span in Future AGI without us writing any tracing code.

The resulting span captures:

- the model and request parameters (`max_tokens`, `tools`),
- the input messages,
- the response content blocks (including the `tool_use` block Claude emits),
- the `stop_reason`,
- the token usage (input and output).

In [ ]:
import anthropic

client = anthropic.Anthropic()

tools = [
    {
        "name": "get_weather",
        "description": "Get the current weather in a given location.",
        "input_schema": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "City and state, e.g. San Francisco, CA",
                },
                "unit": {
                    "type": "string",
                    "enum": ["celsius", "fahrenheit"],
                },
            },
            "required": ["location"],
        },
    }
]

response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=512,
    tools=tools,
    messages=[
        {"role": "user", "content": "What is the weather in San Francisco right now?"}
    ],
)

print("stop_reason:", response.stop_reason)
for block in response.content:
    if block.type == "tool_use":
        print(f"tool_use: {block.name}({block.input})")
    elif block.type == "text":
        print("text:", block.text)

## View the trace in Future AGI

Open the [Future AGI dashboard](https://app.futureagi.com) and navigate to the `anthropic_quickstart` project. The call above appears as a single `messages.create` span with the request, the `tool_use` response, and the token-usage attributes attached. From there you can attach evaluators, build datasets, or compare prompt variants side-by-side.

# Feedback

---

If you have any feedback or requests, please open an issue on the [`traceAI` GitHub repo](https://github.com/future-agi/traceAI/issues) or reach out via [docs.futureagi.com](https://docs.futureagi.com).